# 12c. Joint Optimization: FWHM + Sigma + Mean Matching

**Goal:** jointly optimize μ and γ by matching **both** the FWHM distribution and the fit-uncertainty distribution.

**New:** Added a **mean-matching term** to help REINFORCE push μ to the correct value when per-quantile signals become noisy.

| Gradient | Source | Why |
|---|---|---|
| **μ** (REINFORCE + mean matching) | Per-quantile reward + λ_mean·|FWHM_mean − target_mean| | Mean term gives clean signal when per-quantile gradient dies |
| **γ** (implicit diff + CRLB) | dLoss/dγ = W₁'(FWHM)·dFWHM/dγ + λ·W₁'(σ)·dσ/dγ | Matching both tightens γ constraints |

All model code from `src/`.

In [ ]:
import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

from src.fitting import (
    _raw_from_width as _rw,
    log_pdf, nll, fwhm_from_theta, fit_profile
)
from src.samplers import draw_fixed_noise, build_photons
from src.implicit import compute_fwhm_and_dgamma

print('Imports OK')

In [ ]:
GAMMA_TRUE = 20.0
NBAR_TRUE = 50.0
LAMBDA_ = 2.0

N_TARGET = 200
N_RUNS = 200
N_ITER = 80

SIGMA_PROP = 6.0          # physical noise std
LR_MU = 15.0              # max learning rate for mu (decays to floor)
LR_GAMMA = 0.5            # learning rate for gamma
BASELINE_ALPHA = 0.05     # EMA smoothing
CLIP = 10.0               # gradient clipping

LAMBDA_SIGMA = 0.3        # weight for sigma term in loss/reward
LAMBDA_GAMMA_SIGMA = 0.3  # weight for sigma term in gamma gradient
LAMBDA_MEAN = 0.3         # weight for mean-matching term (new!)

MU_INIT = 8.0
GAMMA_INIT = 5.0
SEED = 42

print('Parameters set')

## Generate Target Data

Collect both FWHM and sigma_FWHM at true (μ=50, γ=20).

In [ ]:
t_total = time.time()

def _fit_fn(ph):
    return fit_profile(ph, n_iters=80, model='lorentzian', uniform_bg=False)
def _fwhm_fn(th):
    return fwhm_from_theta(th, model='lorentzian')
def _nll_fn(th, ph):
    return nll(th, ph, model='lorentzian', uniform_bg=False)

print(f"Generating target ({N_TARGET} runs)...", end=" ", flush=True)
rng = np.random.default_rng(SEED)
target_fwhms, target_sigmas = [], []

for ti in range(N_TARGET):
    if ti % 100 == 0:
        print(f'{ti}...', end=' ', flush=True)
    u, b, n = draw_fixed_noise(NBAR_TRUE, 6, LAMBDA_, rng)
    fw, sig, _ = compute_fwhm_and_dgamma(
        GAMMA_TRUE, u.numpy(), b.numpy(),
        _fit_fn, _fwhm_fn, _nll_fn, n_params=2
    )
    target_fwhms.append(fw)
    target_sigmas.append(sig)

target_t = torch.tensor(target_fwhms, dtype=torch.float32)
target_s_t = torch.tensor(target_sigmas, dtype=torch.float32)
st, _ = torch.sort(target_t)
st_sig, _ = torch.sort(target_s_t)

# Initial forward pass at (MU_INIT, GAMMA_INIT) for visualization
rng_init = np.random.default_rng(999)
init_fwhms = []
for _ in range(500):
    u, b, n = draw_fixed_noise(MU_INIT, SIGMA_PROP, LAMBDA_, rng_init)
    photons = build_photons(torch.tensor(GAMMA_INIT, dtype=torch.float32), u, b)
    theta = fit_profile(photons, n_iters=80, model='lorentzian', uniform_bg=False)
    if theta is not None:
        init_fwhms.append(fwhm_from_theta(theta, model='lorentzian').item())
    else:
        init_fwhms.append(2.0 * GAMMA_INIT)
init_t = torch.tensor(init_fwhms, dtype=torch.float32)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(target_t.numpy(), bins=40, density=True, alpha=0.7, color='#2d6a4f')
ax1.set_xlabel('FWHM (MHz)'); ax1.set_ylabel('Density')
ax1.set_title(f'Target FWHM (μ={NBAR_TRUE}, γ={GAMMA_TRUE})')
ax1.grid(alpha=0.2)
ax2.hist(target_s_t.numpy(), bins=40, density=True, alpha=0.7, color='#2d6a4f')
ax2.set_xlabel('σ_FWHM (MHz)'); ax2.set_ylabel('Density')
ax2.set_title('Target Fit Uncertainty')
ax2.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print(f"FWHM mean={target_t.mean():.1f}, sigma mean={target_s_t.mean():.2f} ({time.time()-t_total:.0f}s)")

## Joint Optimization: FWHM + Sigma + Mean Matching

### μ update (REINFORCE + mean matching)
The combined loss includes:
- Per-quantile FWHM loss (W1)
- Per-quantile sigma loss (W1)
- **Mean FWHM mismatch** (new! gives clean gradient when per-quantile signal is noisy)

```
cl_i = |FWHM_i - target_i| + λ_sig·|σ_i - σ_target_i| + λ_mean·|FWHM_mean - target_mean|
adv_i = cl_i - baseline
∇_μ = mean(adv_i × (n_i − μ) / σ²)
```

In [ ]:
mu_val = float(MU_INIT)
gamma_val = float(GAMMA_INIT)
bl = 0.0
history = []

print(f"μ_init={MU_INIT}, γ_init={GAMMA_INIT}, true=({NBAR_TRUE},{GAMMA_TRUE})")
print(f"N_ITER={N_ITER}, N_RUNS={N_RUNS}")
print(f"λ_sig={LAMBDA_SIGMA}, λ_γ_sig={LAMBDA_GAMMA_SIGMA}, λ_mean={LAMBDA_MEAN}")
print(f"MU LR: linear decay {LR_MU} -> {0.3*LR_MU:.1f} over {N_ITER} steps\n")

for step in range(N_ITER):
    rng2 = np.random.default_rng(SEED + step)
    fwhms, sigmas, dfs, ns = [], [], [], []
    
    for _ in range(N_RUNS):
        u, b, n = draw_fixed_noise(mu_val, SIGMA_PROP, LAMBDA_, rng2)
        ns.append(n)
        fw, sig, dg = compute_fwhm_and_dgamma(
            gamma_val, u.numpy(), b.numpy(),
            _fit_fn, _fwhm_fn, _nll_fn, n_params=2
        )
        fwhms.append(fw)
        sigmas.append(sig)
        dfs.append(dg)
    
    ft = torch.tensor(fwhms, dtype=torch.float32)
    si_t = torch.tensor(sigmas, dtype=torch.float32)
    nt = torch.tensor(ns, dtype=torch.float32)
    dg_t = torch.tensor(dfs, dtype=torch.float32)
    
    # Sorted quantile matching
    sf, sidx = torch.sort(ft)
    pl = torch.abs(sf - st[:N_RUNS])           # FWHM loss per quantile
    nss = nt[sidx]                               # n sorted by FWHM rank
    dgs = dg_t[sidx]                             # dFWHM/dγ sorted by FWHM rank
    ss = si_t[sidx]                              # σ sorted by FWHM rank
    
    # Sigma loss per quantile (same sort order)
    pl_sig = torch.abs(ss - st_sig[:N_RUNS])
    
    # Mean-matching loss (gives clean signal when per-quantile is noisy)
    mean_fwhm_loss = torch.abs(ft.mean() - target_t.mean())
    
    # Combined per-quantile loss with mean matching
    cl = pl + LAMBDA_SIGMA * pl_sig + LAMBDA_MEAN * mean_fwhm_loss
    
    loss_fwhm = pl.mean()
    loss_sigma = pl_sig.mean()
    mean_loss = (loss_fwhm + LAMBDA_SIGMA * loss_sigma + LAMBDA_MEAN * mean_fwhm_loss).item()
    loss_fwhm_val = loss_fwhm.item()
    loss_sigma_val = loss_sigma.item()
    mean_fwhm_val = mean_fwhm_loss.item()
    
    # Baseline
    if step == 0:
        bl = mean_loss
    else:
        bl = (1 - BASELINE_ALPHA) * bl + BASELINE_ALPHA * mean_loss
    
    # ---- MU gradient: REINFORCE with decaying LR (floor=0.3) ----
    lr_mu_decay = LR_MU * max(0.3, 1.0 - step / N_ITER)  # 15 -> 4.5
    adv = (cl.detach() - bl).numpy()
    scores = (nss.numpy() - mu_val) / SIGMA_PROP**2
    raw_grad_mu = float(np.mean(adv * scores))
    grad_mu = max(min(raw_grad_mu, CLIP), -CLIP)
    mu_val += lr_mu_decay * (-grad_mu)
    mu_val = max(1.0, min(200.0, mu_val))
    
    # ---- GAMMA gradient: implicit diff (FWHM) + CRLB (sigma) ----
    dsigs = torch.tensor([min(2.0 / math.sqrt(max(int(n), 1)), 2.0) for n in ns], dtype=torch.float32)
    ds_s = dsigs[sidx]
    
    signs_fw = torch.sign(sf - st[:N_RUNS])
    signs_sg = torch.sign(ss - st_sig[:N_RUNS])
    raw_grad_fwhm = float((signs_fw * dgs).mean().item())
    raw_grad_sigma = float((signs_sg * ds_s).mean().item())
    raw_grad_gamma = raw_grad_fwhm + LAMBDA_GAMMA_SIGMA * raw_grad_sigma
    grad_gamma = max(min(raw_grad_gamma, CLIP), -CLIP)
    gamma_val += LR_GAMMA * (-grad_gamma)
    gamma_val = max(0.1, min(100.0, gamma_val))
    
    # Diagnostics
    rho = float(np.corrcoef(nss.numpy(), pl.numpy())[0, 1]) if pl.std() > 0.01 and nss.std() > 0.01 else 0.0
    
    info = {
        'step': step,
        'mu': mu_val, 'gamma': gamma_val,
        'loss': mean_loss, 'loss_fwhm': loss_fwhm_val, 'loss_sigma': loss_sigma_val,
        'mean_fwhm_diff': mean_fwhm_val,
        'baseline': bl,
        'grad_mu': grad_mu, 'grad_gamma': grad_gamma,
        'rho': rho, 'mean_n': float(np.mean(ns)),
        'mean_fwhm': float(ft.mean().item()),
        'mean_sigma': float(ss.mean().item()),
    }
    history.append(info)
    
    if step % 5 == 0 or step == N_ITER - 1:
        print(f"  S{step:2d}: μ={mu_val:6.2f} γ={gamma_val:5.1f} | "
              f"L={mean_loss:.2f}(F={loss_fwhm_val:.2f}+S={loss_sigma_val:.2f}+M={mean_fwhm_val:.2f}) | "
              f"∇μ={grad_mu:+.4f} ∇γ={grad_gamma:+.4f} | "
              f"n̄={info['mean_n']:4.1f} ρ={rho:+.3f} "
              f"({time.time()-t_total:.0f}s)", flush=True)

print(f"\nDone. {time.time()-t_total:.0f}s")

## Results

In [ ]:
if len(history) > 0:
    fmu = history[-1]['mu']
    fga = history[-1]['gamma']
    print(f"{'='*65}")
    print(f"  JOINT OPTIMIZATION — FWHM + SIGMA + MEAN MATCHING")
    print(f"{'='*65}")
    print(f"  μ:     {MU_INIT:.0f} → {fmu:.2f}  (true={NBAR_TRUE})  error={abs(fmu-NBAR_TRUE):.2f}")
    print(f"  γ:     {GAMMA_INIT:.0f} → {fga:.2f}  (true={GAMMA_TRUE})  error={abs(fga-GAMMA_TRUE):.2f}")
    print(f"  Loss:  {history[0]['loss']:.2f} → {history[-1]['loss']:.2f}")
    print(f"  FWHM:  {history[0]['loss_fwhm']:.2f} → {history[-1]['loss_fwhm']:.2f}")
    print(f"  Sigma: {history[0]['loss_sigma']:.2f} → {history[-1]['loss_sigma']:.2f}")
    print(f"  Mean diff: {history[0].get('mean_fwhm_diff',0):.2f} → {history[-1].get('mean_fwhm_diff',0):.2f}")
    print(f"  μ final grad: {history[-1]['grad_mu']:+.4f}")
    print(f"  Time:  {time.time()-t_total:.0f}s")
    print(f"{'='*65}")

## Visualization for Presentation

### μ and γ convergence + trajectory + FWHM distribution comparison

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from src.losses import wasserstein_loss

steps = [h['step'] for h in history]
mu_hist = [h['mu'] for h in history]
gamma_hist = [h['gamma'] for h in history]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- μ convergence ---
ax = axes[0, 0]
ax.axhline(NBAR_TRUE, color='g', ls='--', linewidth=1.5, alpha=0.7, label=f'True μ={NBAR_TRUE}')
ax.plot(steps, mu_hist, 'b-', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('μ')
ax.set_title('REINFORCE + Mean: μ Convergence'); ax.legend(); ax.grid(alpha=0.3)

# --- γ convergence ---
ax = axes[0, 1]
ax.axhline(GAMMA_TRUE, color='g', ls='--', linewidth=1.5, alpha=0.7, label=f'True γ={GAMMA_TRUE}')
ax.plot(steps, gamma_hist, 'r-', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('γ')
ax.set_title('Implicit Diff: γ Convergence'); ax.legend(); ax.grid(alpha=0.3)

# --- Loss ---
ax = axes[0, 2]
loss_hist = [h['loss'] for h in history]
bl_hist = [h['baseline'] for h in history]
ax.plot(steps, loss_hist, 'k-', linewidth=2, label='Combined loss')
ax.plot(steps, bl_hist, 'k--', linewidth=1.5, alpha=0.6, label='Baseline')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss')
ax.set_title('Optimization Loss'); ax.legend(); ax.grid(alpha=0.3)

# --- μ gradient ---
ax = axes[1, 0]
grad_mu_hist = [h['grad_mu'] for h in history]
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.plot(steps, grad_mu_hist, 'b-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('∇μ')
ax.set_title('REINFORCE μ Gradient'); ax.grid(alpha=0.3)

# --- γ gradient ---
ax = axes[1, 1]
grad_gamma_hist = [h['grad_gamma'] for h in history]
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.plot(steps, grad_gamma_hist, 'r-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('∇γ')
ax.set_title('Implicit γ Gradient'); ax.grid(alpha=0.3)

# --- Trajectory (μ vs γ) ---
ax = axes[1, 2]
ax.plot(mu_hist, gamma_hist, 'b.-', linewidth=1.5, markersize=8)
ax.plot(mu_hist[0], gamma_hist[0], 'go', markersize=10, label=f'Start ({MU_INIT},{GAMMA_INIT})')
ax.plot(mu_hist[-1], gamma_hist[-1], 'ro', markersize=10, label=f'End ({mu_hist[-1]:.1f},{gamma_hist[-1]:.1f})')
ax.plot(NBAR_TRUE, GAMMA_TRUE, 'k*', markersize=15, label=f'True ({NBAR_TRUE},{GAMMA_TRUE})')
ax.set_xlabel('μ (mean photon count)'); ax.set_ylabel('γ (HWHM MHz)')
ax.set_title('Optimization Trajectory'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Joint Optimization: REINFORCE + Mean + Implicit Diff', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('fig_optimization_path.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final forward pass for FWHM distribution comparison
print('Generating final forward pass for distribution comparison...')
rng_f = np.random.default_rng(999)
final_fwhms = []
for _ in range(500):
    u, b, n = draw_fixed_noise(mu_hist[-1], SIGMA_PROP, LAMBDA_, rng_f)
    photons = build_photons(torch.tensor(gamma_hist[-1], dtype=torch.float32), u, b)
    theta = fit_profile(photons, n_iters=80, model='lorentzian', uniform_bg=False)
    if theta is not None:
        final_fwhms.append(fwhm_from_theta(theta, model='lorentzian').item())
    else:
        final_fwhms.append(2.0 * gamma_hist[-1])
final_t = torch.tensor(final_fwhms, dtype=torch.float32)

# Initial vs Target vs Final KDE
x_grid = np.linspace(0, 100, 500)
kde_target = gaussian_kde(target_t.numpy())
kde_initial = gaussian_kde(init_t.numpy())
kde_final = gaussian_kde(final_t.numpy())
w1_final = wasserstein_loss(final_t, target_t).item()
w1_initial = wasserstein_loss(init_t, target_t).item()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_grid, kde_target(x_grid), 'k-', linewidth=2.5, label=f'Target (μ={NBAR_TRUE}, γ={GAMMA_TRUE})')
ax.plot(x_grid, kde_initial(x_grid), 'b--', linewidth=2, label=f'Initial (μ={MU_INIT}, γ={GAMMA_INIT}, W1={w1_initial:.0f})')
ax.plot(x_grid, kde_final(x_grid), 'r-', linewidth=2.5, label=f'Final (μ={mu_hist[-1]:.1f}, γ={gamma_hist[-1]:.1f}, W1={w1_final:.1f})')
ax.fill_between(x_grid, kde_final(x_grid), kde_target(x_grid), alpha=0.12, color='gray')
ax.set_xlabel('FWHM (MHz)'); ax.set_ylabel('Density')
ax.set_title('FWHM Distribution: Optimization Result')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_fwhm_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Target FWHM: {target_t.mean():.1f} ± {target_t.std():.1f}')
print(f'Final  FWHM: {final_t.mean():.1f} ± {final_t.std():.1f}')
print(f'Initial FWHM: {init_t.mean():.1f} ± {init_t.std():.1f}')